## Initializing the functions

In [0]:
%run /Configurations/Init_Scripts

## Declaring the source and destination paths

In [0]:
import requests 
import json

In [0]:
dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","38")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','13')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('RecordTypeId','0123h000000khU7AAI')
RecordTypeId = dbutils.widgets.get('RecordTypeId')

dbutils.widgets.text('CreatedBy','ADB_MoxieShipToAccountFullLoad')
CreatedBy = dbutils.widgets.get('CreatedBy')


dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
RawPath_account=ExternalLocation_raw+'/SalesForce/Moxie/Account'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
SilverPath_account=ExternalLocationName_silver+'/DIMAccount'


In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# RawPath_account = '/mnt/raw/SalesForce/Moxie/Account'
# SilverPath_account = '/mnt/silver/DIMAccount'

API_RETRY_LIMIT = 5

raw_schema = StructType([
StructField("SAP_Account__c", StringType(), True),
StructField("Name", StringType(), True),
StructField("PPP_SCS__c", BooleanType(), True),
StructField("PPP_CTS__c", BooleanType(), True),
StructField("PPP_CSEliteSystem__c", BooleanType(), True),
StructField("PPP_DiamondGlow__c", BooleanType(), True),
StructField("Per_patient_Pricing_program__c", BooleanType(), True),
StructField("P3_Effective_Date__c", StringType(), True),
StructField("Id", StringType(), True)
])

In [0]:
Client_id = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-ClientID')
Client_secret = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Secret')
Username = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-UserName')
Password = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Password')
InstanceURL = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-InstanceURL')
Domain   = dbutils.secrets.get('ABV_AKV_ADB_SCOPE','Moxie-Domain')

In [0]:
#Reading data from DimAccount
#account_hist_df = spark.read.format("delta").load(SilverPath_account)

## Data Ingestion

In [0]:
#Generate Access Token
def generate_token():   
    token_endpoint = f'https://{Domain}/services/oauth2/token' 
    payload = {
        'grant_type': 'password',
        'client_id': Client_id,
        'client_secret': Client_secret,
        'username': Username,
        'password': Password
    }

    retry_count = 0
    while (retry_count <= API_RETRY_LIMIT):
      response = requests.post(token_endpoint, data=payload)
      if(response.status_code == 200): 
          print(f"Successfully retrieved the Token.")
          response_data = response.json()
          return response_data
      elif(500 <= response.status_code <= 599 or response.status_code == 429):
        print(f'url:{url};', f'response_code:{response.status_code};' f'error_message:{response.text}')
        print("Waiting for 90 seconds...")
        sleep(90)
        retry_count += 1
      else:
        raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')
    raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')

In [0]:
#Reading Data from Moxie
def Query(soql_query):
    endpoint = f"https://{InstanceURL}/services/data/v58.0/query/"
    records = []

    headers = {
            'Authorization': f'Bearer {access_token}',
            'Content-Type': 'application/json'
        }

    retry_count = 0
    while (retry_count <= API_RETRY_LIMIT):
      response = requests.get(endpoint, headers=headers, params={'q': f'{soql_query}'})
      if(response.status_code == 200): 
          print(f"Successfully retrieved the Data.")
          response_data = response.json()
          return response_data
      elif(500 <= response.status_code <= 599 or response.status_code == 429):
        print(f'url:{url};', f'response_code:{response.status_code};' f'error_message:{response.text}')
        print("Waiting for 90 seconds...")
        sleep(90)
        retry_count += 1
      else:
        raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')
    raise Exception(f'url:{endpoint};', f'response_code:{response.status_code};' f'error_message:{response.text}')

In [0]:
#Creating Log Entry
log_dict = {
 "ConfigID" : ConfigID,
  "SourceTypeID" : sourceTypeId,
  "Source" : "Moxie",
  "SourceFileName" : "ShipToAccount",
  "Destination" : "rawzone.raw_ShipToAccount",
  "DestinationFileName" : "",
  "Run_ID": run_id,
  "Job_ID": job_id
}
print(log_dict)
audit_df = spark.createDataFrame([log_dict])

## Pulling Data From Moxie

In [0]:
try:
    logIntoPromotionLogTable(audit_df,CreatedBy,"InProgress")
    access_token = generate_token()['access_token']
    print(access_token)

    raw_data = Query(f'''SELECT SAP_Account__c,
                        Name,
                        PPP_SCS__c,
                        PPP_CTS__c,
                        PPP_CSEliteSystem__c,
                        PPP_DiamondGlow__c,
                        Per_patient_Pricing_program__c,
                        P3_Effective_Date__c,
                        Id
                From Account 
                where recordtypeid=\'{RecordTypeId}\' and Per_patient_Pricing_program__c = true''')
    df_raw = spark.createDataFrame(raw_data['records'],raw_schema)\
          .drop("attributes")

    df_account = df_raw.withColumnRenamed("Name","ShipToAccountName")\
                    .withColumnRenamed("SAP_Account__c","ShipToAccountId")\
                    .withColumnRenamed("Per_patient_Pricing_program__c","PerPatientPricingFlag")\
                    .withColumnRenamed('PPP_SCS__c','CoolSculptingMachineFlag')\
                    .withColumnRenamed('PPP_DiamondGlow__c','DermalInfusionFlag')\
                    .withColumnRenamed('PPP_CTS__c','CoolToneMachineFlag')\
                    .withColumnRenamed('PPP_CSEliteSystem__c','CSEliteMachineFlag')\
                    .withColumnRenamed('Id','MoxieShipToAccountId')\
                    .withColumnRenamed('P3_Effective_Date__c','P3_Effective_Begin_Date')\
                    .withColumn('CreatedBy',lit(CreatedBy))\
                    .withColumn('UpdatedBy',lit(CreatedBy))\
                    .withColumn("Createddate", current_timestamp())\
                    .withColumn("Updateddate", current_timestamp())\
                    .withColumn("Active", lit('1').cast('Int'))

    df_account = df_account.select('ShipToAccountId','ShipToAccountName','PerPatientPricingFlag','CoolSculptingMachineFlag','DermalInfusionFlag','CoolToneMachineFlag','CSEliteMachineFlag','MoxieShipToAccountId',col('P3_Effective_Begin_Date').cast("Date"),'CreatedDate','CreatedBy','UpdatedDate','UpdatedBy','Active')

    # #To find unsubscribed Accounts 
    # unsubscribedAccounts_df = account_hist_df.join(account_df,"ShipToAccountId","left_anti")\
    #                                 .withColumn("PerPatientPricingFlag",lit(False))\
    #                                 .withColumn('CreatedBy',lit(CreatedBy))\
    #                                 .withColumn('UpdatedBy',lit(CreatedBy))\
    #                                 .withColumn("Createddate", current_timestamp())\
    #                                 .withColumn("Updateddate", current_timestamp())\
    #                                 .withColumn("Active", lit('0').cast('Int'))\
    #                                 .select('ShipToAccountId','ShipToAccountName','PerPatientPricingFlag','CoolSculptingMachineFlag','DermalInfusionFlag','CoolToneMachineFlag','CSEliteMachineFlag','MoxieShipToAccountId','CreatedDate','CreatedBy','UpdatedDate','UpdatedBy','Active')

    # union_df = unsubscribedAccounts_df.unionByName(account_df,True)
        
    df_account.write.format("delta").mode("append").save(RawPath_account)

    logIntoPromotionLogTable(audit_df,CreatedBy,"Succeeded")
except Exception as e:
    logIntoPromotionLogTable(audit_df,CreatedBy,"Failed",str(e))
    raise e